In [2]:
import pandas as pd
import numpy as np
import joblib as jb
from sklearn.model_selection import train_test_split

In [3]:
house=pd.read_csv('house.csv')
house=house.drop(columns='Unnamed: 0')
house


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,Price
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


In [4]:
target = 'Price'
X = house.drop(target,axis=1).values
y = house[target].values

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.3,
                                                    random_state=667,
                                                    )

In [5]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures

In [6]:
polyfeats = PolynomialFeatures(degree=3)
X_poly = polyfeats.fit_transform(X)

print("Numero di esempi nel test: "+str(X_poly.shape[0]))
print("Numero di features: "+str(X_poly.shape[1]))

Numero di esempi nel test: 506
Numero di features: 560


In [7]:
scaler = StandardScaler()
X_poly_scaled= scaler.fit_transform(X_poly)

In [8]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """ fitto il modello e lo valido con le metrice di regressione
    """
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    # Misurazione errore
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    # Cross validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

    print(f"\nRisultati per {model_name}:")
    print(f"RMSE: {rmse:.2f}")
    print(f"R2 Score: {r2:.3f}")
    print(f"CV R2 Scores: {cv_scores.mean():.3f} (+/- {cv_scores.std() * 2:.3f})")

    return y_pred, r2

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_poly_scaled,y,
                                                    test_size=0.3,
                                                    random_state=667,
                                                    )

In [10]:
def overfit_eval(model, X, y):

    """
    model: il nostro modello predittivo già addestrato
    X: una tupla contenente le prorietà del train set e test set (X_train, X_test)
    y: una tupla contenente target del train set e test set (y_train, y_test)
    """

    y_pred_train = model.predict(X[0])
    y_pred_test = model.predict(X[1])

    mse_train = mean_squared_error(y[0], y_pred_train)
    mse_test = mean_squared_error(y[1], y_pred_test)

    r2_train = r2_score(y[0], y_pred_train)
    r2_test = r2_score(y[1], y_pred_test)

    print("Train set:  MSE="+str(mse_train)+" R2="+str(r2_train))
    print("Test set:  MSE="+str(mse_test)+" R2="+str(r2_test))

In [11]:
from sklearn.linear_model import ElasticNet
alphas=[0.0001,0.001,0.01,0.1,1,10]
for i in alphas:
    elastic = ElasticNet(alpha=i, l1_ratio=0.5)
    print("Alpha="+str(i))
    elastic.fit(X_train, y_train)
    overfit_eval(elastic, (X_train, X_test),(y_train, y_test))
    y_pred_elastic, r2_elastic = evaluate_model(elastic, X_train, X_test, y_train, y_test, "Elastic Net")

Alpha=0.0001


/usr/lib/python3/dist-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.442e+02, tolerance: 2.923e+00
  model = cd_fast.enet_coordinate_descent(


Train set:  MSE=2.405598213254731 R2=0.9708666217601679
Test set:  MSE=13.442715988225649 R2=0.8484855930164211


/usr/lib/python3/dist-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.442e+02, tolerance: 2.923e+00
  model = cd_fast.enet_coordinate_descent(
/usr/lib/python3/dist-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.939e+02, tolerance: 2.166e+00
  model = cd_fast.enet_coordinate_descent(
/usr/lib/python3/dist-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.779e+02, tolerance: 2.363e+00
  model = cd_fast.enet_


Risultati per Elastic Net:
RMSE: 3.67
R2 Score: 0.848
CV R2 Scores: 0.733 (+/- 0.251)
Alpha=0.001


...


Risultati per Elastic Net:
RMSE: 3.83
R2 Score: 0.834
CV R2 Scores: 0.818 (+/- 0.170)
Alpha=0.1


/usr/lib/python3/dist-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.585e+01, tolerance: 2.923e+00
  model = cd_fast.enet_coordinate_descent(


Train set:  MSE=8.311322052354482 R2=0.8993444176628758
Test set:  MSE=20.742931133093737 R2=0.7662040236151155


/usr/lib/python3/dist-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.585e+01, tolerance: 2.923e+00
  model = cd_fast.enet_coordinate_descent(
/usr/lib/python3/dist-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.012e+01, tolerance: 2.166e+00
  model = cd_fast.enet_coordinate_descent(
/usr/lib/python3/dist-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.787e+01, tolerance: 2.363e+00
  model = cd_fast.enet_


Risultati per Elastic Net:
RMSE: 4.55
R2 Score: 0.766
CV R2 Scores: 0.814 (+/- 0.179)
Alpha=1
Train set:  MSE=16.26480084628134 R2=0.8030225527939903
Test set:  MSE=32.390864037632035 R2=0.6349188243918755

Risultati per Elastic Net:
RMSE: 5.69
R2 Score: 0.635
CV R2 Scores: 0.770 (+/- 0.100)
Alpha=10
Train set:  MSE=63.30526858461509 R2=0.23333151642374594
Test set:  MSE=70.9507604666123 R2=0.2003057710548527

Risultati per Elastic Net:
RMSE: 8.42
R2 Score: 0.200
CV R2 Scores: 0.217 (+/- 0.096)


In [12]:
import warnings
warnings.filterwarnings('ignore')


In [13]:
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)

In [14]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(elastic, X_poly_scaled, y)

print(scores)

[0.76634196 0.8022705  0.77549743 0.38183028 0.52923073]


In [15]:
scores.mean()

0.6510341792696157